# LeetCode 1026 — Maximum Difference Between Node and Ancestor bằng Python

Link đề: https://leetcode.com/problems/maximum-difference-between-node-and-ancestor/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def maxAncestorDiff(self, root: Optional[TreeNode]) -> int:
        ...
```

Ghi chú: đáp án là **một số nguyên duy nhất**. Notebook tự dựng `TreeNode`, hàm dựng cây từ danh sách kiểu LeetCode (`build_tree`, theo tầng, `None` là nút trống) và `to_level`. Điểm cốt lõi: với mỗi nút, chỉ cần so với **giá trị nhỏ nhất và lớn nhất** trong các tổ tiên của nó — mang `(min, max)` của đường đi từ gốc xuống (hoặc trả `(min, max)` của cây con lên). Cây có thể sâu tới `5000` nút nên cần chú ý **giới hạn đệ quy** (cách lặp bằng stack an toàn).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong mọi cặp (tổ tiên `a`, hậu duệ `b`), tìm `|a.val − b.val|` lớn nhất.
2. Nắm công cụ: DFS mang `(min, max)` của đường đi, DFS từ dưới lên trả `(min, max)` của cây con, BFS với trạng thái, bản đồ cha.
3. Đi từ **mỗi nút so với mọi tổ tiên (`O(n·h)`)** → **DFS mang `(min, max)` xuống (lặp)** → **đệ quy từ dưới lên** → **BFS với trạng thái `(nút, min, max)`** → **hậu thứ tự lặp với `(min, max)` của cây con**.
4. Hiểu mấu chốt: `max_b |a − b|` trên các tổ tiên `a` của `b` đạt ở **giá trị nhỏ nhất hoặc lớn nhất** của đường đi gốc → cha của `b`: `max(|b − min|, |b − max|)`.
5. Tránh lỗi kinh điển: chỉ so cha–con, chỉ so với gốc, lấy `max − min` toàn cây, chỉ theo dõi `min` (hoặc chỉ `max`), dùng chung `min/max` giữa các nhánh anh em, tính `max − min` của **cây con** (có thể là hai nút anh em chứ không phải tổ tiên–hậu duệ), và đệ quy quá sâu trên cây lệch.

## 2. Đề bài & ví dụ

Cho gốc `root` của cây nhị phân, tìm giá trị lớn nhất `v` sao cho tồn tại hai nút **khác nhau** `a`, `b` với `v = |a.val − b.val|` và `a` là **tổ tiên** của `b`.

Nút `a` là tổ tiên của `b` nếu một con của `a` bằng `b` hoặc một con của `a` là tổ tiên của `b`.

**Ví dụ 1:** `root = [8,3,10,1,6,null,14,null,null,4,7,13]` → `7` (`|8 − 1| = 7`).

**Ví dụ 2:** `root = [1,null,2,null,0,3]` → `3` (`|1 − 2|`, `|1 − 0|`, `|2 − 0|`, `|2 − 3|`... lớn nhất `|0 − 3| = 3`).

**Ràng buộc:** số nút trong `[2, 5000]`; `0 <= Node.val <= 10^5`.

In [1]:
import itertools
import random
import sys
import time
from collections import deque
from typing import List, Optional

sys.setrecursionlimit(30000)           # cây lệch tới 5000 nút cần đệ quy sâu (giới hạn mặc định 1000)


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

    def __repr__(self):
        return f"TreeNode({self.val})"


def build_tree(values):
    """Dựng cây từ danh sách theo tầng kiểu LeetCode (None = không có nút)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def to_level(root):
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
            continue
        out.append(node.val)
        queue.append(node.left)
        queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


def tup(root):
    return None if root is None else (root.val, tup(root.left), tup(root.right))


def from_tuple(t):
    return None if t is None else TreeNode(t[0], from_tuple(t[1]), from_tuple(t[2]))


def size(root):
    n, stack = 0, [root]
    while stack:
        x = stack.pop()
        if x:
            n += 1
            stack.extend([x.left, x.right])
    return n


L1 = [8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13]
L2 = [1, None, 2, None, 0, 3]
r1 = build_tree(L1)
print(tup(r1), to_level(r1) == L1, "| n =", size(r1))
assert to_level(build_tree(L2)) == L2

(8, (3, (1, None, None), (6, (4, None, None), (7, None, None))), (10, None, (14, (13, None, None), None))) True | n = 9


## 3. Kiến thức nền

### 3.1 Tổ tiên và đường đi

Các tổ tiên của `b` là các nút trên đường đi từ gốc tới **cha** của `b`. Ta dùng **bản đồ cha** (`parent[node]`) để liệt kê chúng, hoặc mang thông tin theo đường đi khi duyệt từ gốc xuống.

In [2]:
def parent_map(root):
    parent = {id(root): None}
    order, queue = [], deque([root])
    while queue:
        x = queue.popleft()
        order.append(x)
        for c in (x.left, x.right):
            if c is not None:
                parent[id(c)] = x
                queue.append(c)
    return parent, order


parent, order = parent_map(build_tree(L1))
node13 = next(x for x in order if x.val == 13)
chain, y = [], parent[id(node13)]
while y is not None:
    chain.append(y.val)
    y = parent[id(y)]
print("tổ tiên của nút 13:", chain)

tổ tiên của nút 13: [14, 10, 8]


### 3.2 Chỉ cần `min` và `max` của các tổ tiên

Với nút `b` cố định, `max_a |a − b|` trên các tổ tiên `a` đạt được tại tổ tiên có giá trị **nhỏ nhất** hoặc **lớn nhất** (vì `|a − b|` là hàm lồi theo `a`: cực đại ở biên). Do đó chỉ cần theo dõi `mn`, `mx` của đường đi.

### 3.3 Cây có thể rất sâu

Cây lệch có `5000` nút sâu `5000` > giới hạn đệ quy mặc định `1000` của Python. Ta tăng `sys.setrecursionlimit` trong notebook; lời giải nộp chính dùng **DFS lặp**.

## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1: đường đi `8 → 3 → 1`: tại nút `1`, `mn = 3`, `mx = 8` (tổ tiên `8, 3`), `|1 − 8| = 7` — lớn nhất. Đường `8 → 10 → 14 → 13`: tại `13`: tổ tiên `8, 10, 14`: `mn = 8`, `mx = 14`: `|13 − 8| = 5`, `|13 − 14| = 1`.

Thay vì xét mọi cặp, mỗi nút chỉ so với `mn` và `mx` của đường đi từ gốc: `O(1)` mỗi nút ⇒ `O(n)`.

In [3]:
def trace_paths(root):
    best = [0]
    stack = [(root, root.val, root.val, [root.val])]
    while stack:
        node, mn, mx, path = stack.pop()
        for c in (node.left, node.right):
            if c is not None:
                d = max(abs(c.val - mn), abs(c.val - mx))
                if d > best[0]:
                    best[0] = d
                    print(f"  nút {c.val} (tổ tiên {path}): min={mn}, max={mx}, |{c.val}-{mn}|={abs(c.val - mn)}, |{c.val}-{mx}|={abs(c.val - mx)} -> kỷ lục mới {d}")
                stack.append((c, min(mn, c.val), max(mx, c.val), path + [c.val]))
    return best[0]


print("đáp án:", trace_paths(build_tree(L1)))

  nút 3 (tổ tiên [8]): min=8, max=8, |3-8|=5, |3-8|=5 -> kỷ lục mới 5
  nút 14 (tổ tiên [8, 10]): min=8, max=10, |14-8|=6, |14-10|=4 -> kỷ lục mới 6
  nút 1 (tổ tiên [8, 3]): min=3, max=8, |1-3|=2, |1-8|=7 -> kỷ lục mới 7
đáp án: 7


## 5. Brute force — mỗi nút so với mọi tổ tiên

**Ý tưởng:** lập bản đồ cha; với mỗi nút đi ngược lên gốc, tính `|nút − tổ tiên|` cho từng tổ tiên và lấy max. `O(n·h)` (xấu nhất `O(n²)`), không đệ quy, độc lập với các cách sau ⇒ **oracle**.

In [4]:
def diff_brute(root):
    parent, order = parent_map(root)
    best = 0
    for x in order:
        y = parent[id(x)]
        while y is not None:
            best = max(best, abs(x.val - y.val))
            y = parent[id(y)]
    return best


assert diff_brute(build_tree(L1)) == 7 and diff_brute(build_tree(L2)) == 3
print(diff_brute(build_tree(L1)), diff_brute(build_tree(L2)))

7 3


In [5]:
# Code nộp LeetCode cho Brute force (mỗi nút so với mọi tổ tiên; O(n·h))
class SolutionBrute:
    def maxAncestorDiff(self, root: Optional[TreeNode]) -> int:
        parent = {root: None}
        queue = [root]
        for x in queue:
            for c in (x.left, x.right):
                if c:
                    parent[c] = x
                    queue.append(c)
        best = 0
        for x in queue:
            y = parent[x]
            while y:
                best = max(best, abs(x.val - y.val))
                y = parent[y]
        return best

**Độ phức tạp:** `O(n·h)` thời gian (`≤ 12.5·10^6` bước với `n = 5000`), `O(n)` bộ nhớ.

**Nút thắt:** với mỗi nút ta duyệt lại cả chuỗi tổ tiên, trong khi chỉ `min` và `max` của chuỗi đó mới quan trọng.

## 6. Bước đột phá

Mang `(mn, mx)` của các tổ tiên xuống: khi đi từ nút `u` tới con `c`, `mn' = min(mn, u.val)`, `mx' = max(mx, u.val)`, và ứng viên đáp án là `max(|c.val − mn'|, |c.val − mx'|)`. Mỗi nút `O(1)` ⇒ `O(n)`.

## 7. Các cách tiếp theo

Ngoài Brute force (mục 5), ta có thêm 4 cách: **Cách 2** DFS lặp mang `(min, max)` xuống (đáp án chính), **Cách 3** đệ quy từ dưới lên, **Cách 4** BFS với trạng thái, **Cách 5** hậu thứ tự lặp với `(min, max)` của cây con.

### Cách 2 — DFS lặp mang `(min, max)` của đường đi

**Giải thích từng đoạn:**

1. Stack chứa `(nút, mn, mx)` với `mn`, `mx` là min/max giá trị trên đường đi từ gốc tới nút (**gồm cả nút**).
2. Với mỗi nút lấy ra: với từng con `c`: ứng viên `max(|c.val − mn|, |c.val − mx|)`; đẩy `(c, min(mn, c.val), max(mx, c.val))`.

In [6]:
def diff_dfs_iter(root):
    best = 0
    stack = [(root, root.val, root.val)]
    while stack:
        node, mn, mx = stack.pop()
        for c in (node.left, node.right):
            if c is not None:
                best = max(best, abs(c.val - mn), abs(c.val - mx))      # so với min/max của TỔ TIÊN của c
                stack.append((c, min(mn, c.val), max(mx, c.val)))
    return best


assert diff_dfs_iter(build_tree(L1)) == 7 and diff_dfs_iter(build_tree(L2)) == 3
print(diff_dfs_iter(build_tree(L1)), diff_dfs_iter(build_tree(L2)))

7 3


In [7]:
# Code nộp LeetCode cho Cách 2 (an toàn với cây lệch 5000 nút)
class SolutionDFSIter:
    def maxAncestorDiff(self, root: Optional[TreeNode]) -> int:
        best = 0
        stack = [(root, root.val, root.val)]
        while stack:
            node, mn, mx = stack.pop()
            for c in (node.left, node.right):
                if c:
                    best = max(best, abs(c.val - mn), abs(c.val - mx))
                    stack.append((c, min(mn, c.val), max(mx, c.val)))
        return best

**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ (stack trường hợp xấu).

### Cách 3 — Đệ quy từ dưới lên

**Ý tưởng:** `go(node)` trả `(mn, mx)` của **cây con** gốc `node`. Tổ tiên `node` so với mọi hậu duệ của nó: `max(|node.val − mn_con|, |node.val − mx_con|)`; cập nhật đáp án; trả `(min(mn_con, node.val), max(mx_con, node.val))`.

In [8]:
def diff_bottom_up(root):
    best = 0

    def go(node):
        nonlocal best
        mn = mx = node.val
        for c in (node.left, node.right):
            if c is not None:
                cmn, cmx = go(c)
                best = max(best, abs(node.val - cmn), abs(node.val - cmx))   # node là tổ tiên của mọi nút trong cây con của c
                mn, mx = min(mn, cmn), max(mx, cmx)
        return mn, mx

    go(root)
    return best


assert diff_bottom_up(build_tree(L1)) == 7 and diff_bottom_up(build_tree(L2)) == 3
print(diff_bottom_up(build_tree(L1)), diff_bottom_up(build_tree(L2)))

7 3


In [9]:
# Code nộp LeetCode cho Cách 3 (đệ quy; cây lệch sâu có thể vượt giới hạn đệ quy mặc định)
class SolutionBottomUp:
    def maxAncestorDiff(self, root: Optional[TreeNode]) -> int:
        best = 0

        def go(node):
            nonlocal best
            mn = mx = node.val
            for c in (node.left, node.right):
                if c:
                    cmn, cmx = go(c)
                    best = max(best, abs(node.val - cmn), abs(node.val - cmx))
                    mn, mx = min(mn, cmn), max(mx, cmx)
            return mn, mx

        go(root)
        return best

**Độ phức tạp:** `O(n)` thời gian, `O(h)` ngăn xếp đệ quy.

### Cách 4 — BFS với trạng thái `(nút, mn, mx)`

**Ý tưởng:** như Cách 2 nhưng dùng hàng đợi; thứ tự duyệt không ảnh hưởng vì mỗi nút mang riêng `(mn, mx)` của đường đi của nó.

In [10]:
def diff_bfs(root):
    best = 0
    queue = deque([(root, root.val, root.val)])
    while queue:
        node, mn, mx = queue.popleft()
        for c in (node.left, node.right):
            if c is not None:
                best = max(best, abs(c.val - mn), abs(c.val - mx))
                queue.append((c, min(mn, c.val), max(mx, c.val)))
    return best


assert diff_bfs(build_tree(L1)) == 7 and diff_bfs(build_tree(L2)) == 3
print(diff_bfs(build_tree(L1)), diff_bfs(build_tree(L2)))

7 3


In [11]:
# Code nộp LeetCode cho Cách 4
from collections import deque


class SolutionBFS:
    def maxAncestorDiff(self, root: Optional[TreeNode]) -> int:
        best = 0
        queue = deque([(root, root.val, root.val)])
        while queue:
            node, mn, mx = queue.popleft()
            for c in (node.left, node.right):
                if c:
                    best = max(best, abs(c.val - mn), abs(c.val - mx))
                    queue.append((c, min(mn, c.val), max(mx, c.val)))
        return best

**Độ phức tạp:** `O(n)` thời gian, `O(w)` bộ nhớ (`w` chiều rộng lớn nhất).

### Cách 5 — Hậu thứ tự lặp với `(min, max)` của cây con

**Ý tưởng:** cùng công thức Cách 3 nhưng **không đệ quy**: duyệt hậu thứ tự bằng stack hai pha; lưu `(mn, mx)` của cây con trong từ điển khoá `id(nút)`; khi xử lý nút (sau con), tổ hợp từ hai con.

In [12]:
def diff_postorder_iter(root):
    best = 0
    sub = {}                                           # id(nút) -> (mn, mx) của cây con
    stack = [(root, False)]
    while stack:
        node, done = stack.pop()
        if not done:
            stack.append((node, True))
            for c in (node.left, node.right):
                if c is not None:
                    stack.append((c, False))
        else:
            mn = mx = node.val
            for c in (node.left, node.right):
                if c is not None:
                    cmn, cmx = sub.pop(id(c))
                    best = max(best, abs(node.val - cmn), abs(node.val - cmx))
                    mn, mx = min(mn, cmn), max(mx, cmx)
            sub[id(node)] = (mn, mx)
    return best


assert diff_postorder_iter(build_tree(L1)) == 7 and diff_postorder_iter(build_tree(L2)) == 3
print(diff_postorder_iter(build_tree(L1)), diff_postorder_iter(build_tree(L2)))

7 3


In [13]:
# Code nộp LeetCode cho Cách 5
class SolutionPostIter:
    def maxAncestorDiff(self, root: Optional[TreeNode]) -> int:
        best = 0
        sub = {}
        stack = [(root, False)]
        while stack:
            node, done = stack.pop()
            if not done:
                stack.append((node, True))
                for c in (node.left, node.right):
                    if c:
                        stack.append((c, False))
            else:
                mn = mx = node.val
                for c in (node.left, node.right):
                    if c:
                        cmn, cmx = sub.pop(c)
                        best = max(best, abs(node.val - cmn), abs(node.val - cmx))
                        mn, mx = min(mn, cmn), max(mx, cmx)
                sub[node] = (mn, mx)
        return best

**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ (từ điển các cây con đang chờ).

## 8. TRACE từng bước

**Trace Cách 2** với `root = [8,3,10,1,6,null,14,null,null,4,7,13]`. Cây: `8(3(1,6(4,7)), 10(·,14(13,·)))`.

| nút (đường đi) | `mn` | `mx` | ứng viên `max(\|v−mn\|,\|v−mx\|)` |
|---|---|---|---|
| 3 (`8`) | 8 | 8 | 5 |
| 10 (`8`) | 8 | 8 | 2 |
| 1 (`8,3`) | 3 | 8 | `max(2, 7) = 7` |
| 6 (`8,3`) | 3 | 8 | `max(3, 2) = 3` |
| 14 (`8,10`) | 8 | 10 | `max(6, 4) = 6` |
| 4 (`8,3,6`) | 3 | 8 | `max(1, 4) = 4` |
| 7 (`8,3,6`) | 3 | 8 | `max(4, 1) = 4` |
| 13 (`8,10,14`) | 8 | 14 | `max(5, 1) = 5` |

Lớn nhất `7` ✓ (tại nút `1`). Hàm ở mục 4 (`trace_paths`) in các kỷ lục mới.

**Trace trường hợp "không phải tổ tiên":** cây `[5, 0, 10]`: hai nút `0` và `10` là **anh em**, hiệu `10` nhưng **không** phải tổ tiên–hậu duệ; đáp án đúng `max(|5−0|, |5−10|) = 5`.

In [14]:
t = build_tree([5, 0, 10])
assert diff_dfs_iter(t) == 5 == diff_brute(t)
print("[5,0,10] ->", diff_dfs_iter(t), "(không phải 10 = |0 - 10| của hai nút anh em)")

[5,0,10] -> 5 (không phải 10 = |0 - 10| của hai nút anh em)


## 9. Code nộp LeetCode

Chọn **Cách 2** (DFS lặp mang `(min, max)`): `O(n)` và an toàn với cây lệch. Giữ nguyên `class Solution`, `maxAncestorDiff(self, root)`:

In [15]:
class Solution:
    def maxAncestorDiff(self, root: Optional[TreeNode]) -> int:
        best = 0
        stack = [(root, root.val, root.val)]           # (nút, min, max) của đường đi gốc -> nút (gồm cả nút)
        while stack:
            node, mn, mx = stack.pop()
            for child in (node.left, node.right):
                if child:
                    best = max(best, abs(child.val - mn), abs(child.val - mx))   # so với tổ tiên nhỏ/lớn nhất
                    stack.append((child, min(mn, child.val), max(mx, child.val)))
        return best


print(Solution().maxAncestorDiff(build_tree(L1)), Solution().maxAncestorDiff(build_tree(L2)))
assert Solution().maxAncestorDiff(build_tree(L1)) == 7 and Solution().maxAncestorDiff(build_tree(L2)) == 3

7 3


## 10. Vì sao đúng

**(a) Chỉ cần `min` và `max` của tổ tiên.** Với nút `b` cố định và tập tổ tiên `A`, hàm `f(a) = |a − b|` là lồi theo `a`, nên `max_{a∈A} f(a)` đạt tại `a = min A` hoặc `a = max A`. Do đó `max_{a∈A} |a − b| = max(|b − min A|, |b − max A|)`.

**(b) Cách 2.** Bất biến: khi lấy nút `u` ra khỏi stack, `(mn, mx)` là min/max của **mọi giá trị trên đường đi từ gốc tới `u` (gồm `u`)**. Với con `c` của `u`, tập tổ tiên của `c` chính là các nút trên đường đi gốc → `u`, nên ứng viên của `c` là `max(|c − mn|, |c − mx|)`; và đường đi tới `c` có `(min(mn, c), max(mx, c))`. Mỗi nút (trừ gốc) được xét đúng một lần, nên đáp án là max trên mọi hậu duệ `b` của `max_a |a − b|`.

**(c) Cách 3 (dưới lên).** Gọi `S(u)` là cây con gốc `u`. Mọi cặp (tổ tiên `a`, hậu duệ `b`) có `b ∈ S(a) \ {a}`. Với nút `a` và mỗi con `c`: `max_{b∈S(c)} |a − b| = max(|a − min S(c)|, |a − max S(c)|)` (cùng lập luận lồi). Do đó `best = max_a max_c (...)` chính là đáp án.

**(d) Cách 4, 5** là biến thể duyệt (BFS / hậu thứ tự lặp) của cùng hai công thức.

**(e) Tại sao `max − min` toàn cây / cây con sai.** Hai nút có giá trị chênh lệch lớn nhất có thể là **anh em** hoặc họ hàng ngang (không phải tổ tiên–hậu duệ), nên đáp án thực tế có thể nhỏ hơn.

**(f) Kiểm chứng bằng máy.** Với **mọi** cây `2..5` nút nhãn `{0, 3, 10}` (`11 493` cây), mọi cách khớp hai oracle độc lập (bản đồ cha và đệ quy liệt kê cặp trên bộ lồng nhau); khớp nhau trên cây ngẫu nhiên tới `5000` nút (kể cả cây lệch sâu `5000`) (mục 12).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Mọi tổ tiên | `O(n·h)` | `O(n)` | oracle |
| 2. DFS lặp `(min, max)` | `O(n)` | `O(n)` | **đáp án chính** |
| 3. Đệ quy dưới lên | `O(n)` | `O(h)` | đệ quy sâu `≤ 5000` |
| 4. BFS | `O(n)` | `O(w)` | theo tầng |
| 5. Hậu thứ tự lặp | `O(n)` | `O(n)` | không đệ quy |

Đo thực tế với cây `5000` nút (ngẫu nhiên, lệch, đầy đủ):

In [16]:
def random_tree(rng, n, shape="random", maxval=10 ** 5):
    """Cây n nút (dựng LẶP). shape: random | chain | complete | comb."""
    vals = [rng.randint(0, maxval) for _ in range(n)]
    nodes = [TreeNode(vals[0])]
    free = [(0, 0), (0, 1)]
    for i in range(1, n):
        node = TreeNode(vals[i])
        nodes.append(node)
        if shape == "random":
            k = rng.randrange(len(free))
            free[k], free[-1] = free[-1], free[k]
            p, side = free.pop()
        elif shape == "chain":
            p, side = i - 1, rng.randrange(2)
        elif shape == "complete":
            p, side = (i - 1) // 2, (i - 1) % 2
        elif shape == "comb":
            p, side = (i - 1 if i % 2 == 1 else i - 2), (0 if i % 2 == 1 else 1)
        else:
            raise ValueError(shape)
        if side == 0:
            nodes[p].left = node
        else:
            nodes[p].right = node
        if shape == "random":
            free.extend([(i, 0), (i, 1)])
    return nodes[0]


def depth(root):
    best, stack = 0, [(root, 1)]
    while stack:
        x, d = stack.pop()
        if x:
            best = max(best, d)
            stack.extend([(x.left, d + 1), (x.right, d + 1)])
    return best


def bench(fn, root, repeat=5):
    best = 1e9
    for _ in range(3):
        t0 = time.perf_counter()
        for _ in range(repeat):
            fn(root)
        best = min(best, (time.perf_counter() - t0) / repeat)
    return best


rng = random.Random(1026)
for shape in ("random", "complete", "chain"):
    tr = random_tree(rng, 5000, shape)
    print(f"{shape:9s}: n = {size(tr)}, độ sâu = {depth(tr)}")
    for name, fn in [("dfs_iter", diff_dfs_iter), ("bottom_up", diff_bottom_up), ("bfs", diff_bfs), ("post_iter", diff_postorder_iter), ("brute", diff_brute)]:
        print(f"    {name:10s}: {bench(fn, tr, repeat=1 if name == 'brute' else 5) * 1e3:8.2f} ms")

random   : n = 5000, độ sâu = 27
    dfs_iter  :     1.59 ms
    bottom_up :     1.75 ms
    bfs       :     1.86 ms


    post_iter :     4.95 ms


    brute     :    11.41 ms


complete : n = 5000, độ sâu = 13
    dfs_iter  :     1.69 ms


    bottom_up :     1.51 ms
    bfs       :     1.54 ms
    post_iter :     2.98 ms
    brute     :     7.42 ms
chain    : n = 5000, độ sâu = 5000


    dfs_iter  :     1.55 ms


    bottom_up :     2.33 ms
    bfs       :     1.49 ms


    post_iter :     3.03 ms


    brute     :  1557.87 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn mọi cây `2..5` nút nhãn `{0, 3, 10}`, (E) tính chất / ngẫu nhiên (cây tới 5000 nút), (F) giới hạn.

### A. Ví dụ của đề

In [17]:
IMPLS = {
    "dfs_iter": diff_dfs_iter,
    "bottom_up": diff_bottom_up,
    "bfs": diff_bfs,
    "post_iter": diff_postorder_iter,
    "brute": diff_brute,
    "Solution": lambda r: Solution().maxAncestorDiff(r),
    "SolutionDFSIter": lambda r: SolutionDFSIter().maxAncestorDiff(r),
    "SolutionBottomUp": lambda r: SolutionBottomUp().maxAncestorDiff(r),
    "SolutionBFS": lambda r: SolutionBFS().maxAncestorDiff(r),
    "SolutionPostIter": lambda r: SolutionPostIter().maxAncestorDiff(r),
    "SolutionBrute": lambda r: SolutionBrute().maxAncestorDiff(r),
}


def oracle_tuple(t):
    """Oracle thứ hai: đệ quy trên bộ lồng nhau — liệt kê mọi cặp (tổ tiên, hậu duệ)."""
    def pairs(node, anc):
        if node is None:
            return 0
        v, l, r = node
        best = max([abs(v - a) for a in anc], default=0)
        return max(best, pairs(l, anc + [v]), pairs(r, anc + [v]))

    return pairs(t, [])


def check(l, want=None):
    root = build_tree(l)
    if want is None:
        want = diff_brute(root)
        if size(root) <= 300:
            assert want == oracle_tuple(tup(root))
    for name in IMPLS:
        assert IMPLS[name](build_tree(l)) == want, (name, l[:10], IMPLS[name](build_tree(l)), want)
    return want


check(L1, 7)
check(L2, 3)
print("ví dụ đề: hai oracle và", len(IMPLS), "cài đặt đúng")

ví dụ đề: hai oracle và 11 cài đặt đúng


### B. Edge case

- Hai nút: `[1, 5]`, `[0, 100000]`, `[5, 5]` (hiệu 0).
- Toàn giá trị bằng nhau ⇒ `0`.
- Chuỗi (mỗi nút một con) tăng/giảm.
- Cây có hai nhánh anh em chênh lệch lớn nhưng tổ tiên chung gần (đáp án nhỏ hơn `max − min`).
- Tổ tiên xa: gốc nhỏ nhất, lá xa lớn nhất.
- Giá trị `0` và `10^5` xen kẽ.
- Cây có một nút ở giữa là cực đại/cực tiểu.

In [18]:
edge = [
    [1, 5], [0, 100000], [5, 5], [5, 5, 5, 5], [5, 0, 10], [0, 5, None, 100], [3, 10, 0], [10, None, 0, None, 100000], [1, 2, None, 3, None, 4], [4, 3, None, 2, None, 1],
    [50, 0, 100, None, 40, None, 60], [0, 100000, 0, 100000, None, None, 100000], [2, 5, 0, None, None, 1], L1, L2,
    [100000, 0, None, 100000, None, 0], [1, None, 2, None, 3, None, 4, None, 5],
]
for l in edge:
    check(l)
assert Solution().maxAncestorDiff(build_tree([5, 0, 10])) == 5
assert Solution().maxAncestorDiff(build_tree([2, 5, 0, None, None, 1])) == 3
print(len(edge), "edge case: mọi cài đặt khớp oracle")

17 edge case: mọi cài đặt khớp oracle


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [19]:
def bug_parent_child_only(root):                        # chỉ so nút với cha
    best, stack = 0, [root]
    while stack:
        node = stack.pop()
        for c in (node.left, node.right):
            if c:
                best = max(best, abs(node.val - c.val))
                stack.append(c)
    return best


def bug_root_only(root):                                # chỉ so với gốc
    best, stack = 0, [root]
    while stack:
        node = stack.pop()
        best = max(best, abs(node.val - root.val))
        stack.extend(c for c in (node.left, node.right) if c)
    return best


def bug_global_range(root):                             # max - min toàn cây
    vals, stack = [], [root]
    while stack:
        node = stack.pop()
        vals.append(node.val)
        stack.extend(c for c in (node.left, node.right) if c)
    return max(vals) - min(vals)


def bug_only_max(root):                                 # chỉ theo dõi max của đường đi
    best, stack = 0, [(root, root.val)]
    while stack:
        node, mx = stack.pop()
        for c in (node.left, node.right):
            if c:
                best = max(best, abs(c.val - mx))
                stack.append((c, max(mx, c.val)))
    return best


def bug_shared_state(root):                             # min/max dùng chung giữa các nhánh anh em (không khôi phục)
    best = 0
    state = {"mn": root.val, "mx": root.val}

    def go(node):
        nonlocal best
        for c in (node.left, node.right):
            if c:
                best = max(best, abs(c.val - state["mn"]), abs(c.val - state["mx"]))
                state["mn"], state["mx"] = min(state["mn"], c.val), max(state["mx"], c.val)    # BUG: không khôi phục khi quay lui
                go(c)

    go(root)
    return best


def bug_subtree_range(root):                            # max(max - min) của mọi cây con (có thể là hai nút anh em)
    best = 0

    def go(node):
        nonlocal best
        mn = mx = node.val
        for c in (node.left, node.right):
            if c:
                a, b = go(c)
                mn, mx = min(mn, a), max(mx, b)
        best = max(best, mx - mn)
        return mn, mx

    go(root)
    return best


def bug_include_self_in_range(root):                    # so nút với min/max đã gồm chính nó (hiệu bằng 0 vô nghĩa) và bỏ qua tổ tiên xa: chỉ min của đường đi
    best, stack = 0, [(root, root.val)]
    while stack:
        node, mn = stack.pop()
        for c in (node.left, node.right):
            if c:
                best = max(best, abs(c.val - mn))
                stack.append((c, min(mn, c.val)))
    return best


BUGS = [bug_parent_child_only, bug_root_only, bug_global_range, bug_only_max, bug_shared_state, bug_subtree_range, bug_include_self_in_range]
probe = [L1, L2, [5, 0, 10], [8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13], [1, 2, None, 3, None, 4], [50, 0, 100, None, 40, None, 60], [2, 5, 0, None, None, 1],
         [10, 4, 7, 20, None, 1], [3, 10, 0], [5, 9, 1, 2, 100, 3, 8]]
for bug in BUGS:
    found = None
    for l in probe:
        try:
            got = bug(build_tree(l))
        except Exception as e:
            got = type(e).__name__
        if got != diff_brute(build_tree(l)):
            found = (l, got)
            break
    assert found, bug.__name__
    print(f"{bug.__name__:28s} sai trên {found[0]}: ra {found[1]}, đúng {diff_brute(build_tree(found[0]))}")

bug_parent_child_only        sai trên [8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13]: ra 5, đúng 7
bug_root_only                sai trên [1, None, 2, None, 0, 3]: ra 2, đúng 3
bug_global_range             sai trên [8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13]: ra 13, đúng 7
bug_only_max                 sai trên [1, None, 2, None, 0, 3]: ra 2, đúng 3
bug_shared_state             sai trên [8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13]: ra 13, đúng 7
bug_subtree_range            sai trên [8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13]: ra 13, đúng 7
bug_include_self_in_range    sai trên [8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13]: ra 6, đúng 7


### D. Vét cạn mọi cây `2..5` nút nhãn `{0, 3, 10}`

`all_trees(n, alphabet)` sinh mọi cây `n` nút: chọn nhãn gốc, chia `n−1` nút còn lại cho trái/phải theo mọi cách, đệ quy. Số cây = Catalan(n) × 3ⁿ: `18 + 135 + 1134 + 10206 = 11 493`. Hai oracle (bản đồ cha / đệ quy bộ lồng nhau) phải khớp nhau và mọi cài đặt khớp chúng.

In [20]:
def all_trees(n, alphabet):
    if n == 0:
        return [None]
    out = []
    for k in range(n):
        for lt in all_trees(k, alphabet):
            for rt in all_trees(n - 1 - k, alphabet):
                for v in alphabet:
                    out.append((v, lt, rt))
    return out


t0 = time.perf_counter()
total = 0
for n in range(2, 6):
    trees = all_trees(n, (0, 3, 10))
    for t in trees:
        want = oracle_tuple(t)
        root = from_tuple(t)
        assert diff_brute(root) == want, (t, want)
        for name in IMPLS:
            assert IMPLS[name](root) == want, (name, t, IMPLS[name](root), want)
    total += len(trees)
    print(f"  n = {n}: {len(trees)} cây")
print(total, "cây:", len(IMPLS), "cài đặt khớp hai oracle;", round(time.perf_counter() - t0, 1), "s")
assert total == 18 + 135 + 1134 + 10206

  n = 2: 18 cây
  n = 3: 135 cây
  n = 4: 1134 cây


  n = 5: 10206 cây
11493 cây: 11 cài đặt khớp hai oracle; 0.4 s


In [21]:
# phản ví dụ nhỏ nhất của từng bug trên không gian vét cạn
for bug in BUGS:
    found = None
    for n in range(2, 6):
        for t in all_trees(n, (0, 3, 10)):
            if bug(from_tuple(t)) != oracle_tuple(t):
                found = t
                break
        if found:
            break
    assert found, bug.__name__
    print(f"{bug.__name__:28s}: cây nhỏ nhất {to_level(from_tuple(found))} -> ra {bug(from_tuple(found))}, đúng {oracle_tuple(found)}")

bug_parent_child_only       : cây nhỏ nhất [10, None, 3, None, 0] -> ra 7, đúng 10
bug_root_only               : cây nhỏ nhất [3, None, 10, None, 0] -> ra 7, đúng 10
bug_global_range            : cây nhỏ nhất [3, 0, 10] -> ra 10, đúng 7
bug_only_max                : cây nhỏ nhất [3, None, 0, None, 10] -> ra 7, đúng 10
bug_shared_state            : cây nhỏ nhất [3, 0, 10] -> ra 10, đúng 7
bug_subtree_range           : cây nhỏ nhất [3, 0, 10] -> ra 10, đúng 7
bug_include_self_in_range   : cây nhỏ nhất [10, None, 3, None, 0] -> ra 7, đúng 10


### E. Tính chất / ngẫu nhiên (cây tới 5000 nút)

(1) Các cách khớp nhau; với `n ≤ 1500` khớp brute; (2) **đáp án `≤ max − min`** của toàn cây; (3) với cây không đối xứng, thêm một lá không làm đáp án giảm; (4) đổi chỗ hai con ở mọi nút (gương) không đổi đáp án; (5) dịch mọi giá trị cộng hằng không đổi đáp án; (6) co giãn `×c` nhân đáp án với `c`; (7) có một cặp (tổ tiên, hậu duệ) đạt đúng đáp án (truy vết).

In [22]:
def mirror(root):
    new_root = TreeNode(root.val)
    stack = [(root, new_root)]
    while stack:
        a, b = stack.pop()
        if a.left:
            b.right = TreeNode(a.left.val)
            stack.append((a.left, b.right))
        if a.right:
            b.left = TreeNode(a.right.val)
            stack.append((a.right, b.left))
    return new_root


def clone(root, f=lambda v: v):
    new_root = TreeNode(f(root.val))
    stack = [(root, new_root)]
    while stack:
        a, b = stack.pop()
        if a.left:
            b.left = TreeNode(f(a.left.val))
            stack.append((a.left, b.left))
        if a.right:
            b.right = TreeNode(f(a.right.val))
            stack.append((a.right, b.right))
    return new_root


def all_values(root):
    out, stack = [], [root]
    while stack:
        x = stack.pop()
        if x:
            out.append(x.val)
            stack.extend([x.left, x.right])
    return out


def best_pair(root):
    """Truy vết một cặp (tổ tiên, hậu duệ) đạt đáp án (giá trị)."""
    parent, order = parent_map(root)
    best, pair = -1, None
    for x in order:
        y = parent[id(x)]
        while y is not None:
            d = abs(x.val - y.val)
            if d > best:
                best, pair = d, (y.val, x.val)
            y = parent[id(y)]
    return best, pair


rng = random.Random(20240115)
t0 = time.perf_counter()
fast = ["dfs_iter", "bottom_up", "bfs", "post_iter", "Solution", "SolutionDFSIter", "SolutionBottomUp", "SolutionBFS", "SolutionPostIter"]
for it in range(120):
    n = rng.choice([2, 3, 5, 10, 50, 200, 1000, 1500, 5000])
    shape = rng.choice(["random", "chain", "complete", "comb"])
    maxval = rng.choice([1, 5, 100, 10 ** 5])
    tree = random_tree(rng, n, shape, maxval)
    res = {name: IMPLS[name](tree) for name in fast}
    assert len(set(res.values())) == 1, (shape, n, res)                                  # (1)
    ans = res["Solution"]
    if n <= 1500:
        assert ans == diff_brute(tree)                                                   # (1)
        a, pair = best_pair(tree)
        assert a == ans and abs(pair[0] - pair[1]) == ans                                # (7)
    vals = all_values(tree)
    assert ans <= max(vals) - min(vals)                                                  # (2)
    assert Solution().maxAncestorDiff(mirror(tree)) == ans                               # (4)
    c = rng.randint(1, 1000)
    assert Solution().maxAncestorDiff(clone(tree, lambda v: v + c)) == ans              # (5)
    assert Solution().maxAncestorDiff(clone(tree, lambda v: v * 3)) == 3 * ans          # (6)
print("120 cây ngẫu nhiên (n tới 5000, 4 hình dạng): 6 tính chất đúng;", round(time.perf_counter() - t0, 1), "s")

120 cây ngẫu nhiên (n tới 5000, 4 hình dạng): 6 tính chất đúng; 3.2 s


In [23]:
# cây nhỏ ngẫu nhiên: so với oracle bộ lồng nhau cho mọi cài đặt
rng = random.Random(102602)
t0 = time.perf_counter()
for _ in range(1500):
    n = rng.randint(2, 40)
    tree = random_tree(rng, n, rng.choice(["random", "chain", "complete", "comb"]), rng.choice([2, 10, 100000]))
    want = oracle_tuple(tup(tree))
    for name in IMPLS:
        assert IMPLS[name](tree) == want, (name, to_level(tree))
print("1500 cây nhỏ ngẫu nhiên: mọi cài đặt khớp oracle đệ quy;", round(time.perf_counter() - t0, 1), "s")

1500 cây nhỏ ngẫu nhiên: mọi cài đặt khớp oracle đệ quy; 0.3 s


### F. Giới hạn

`n = 5000`: cây lệch (chuỗi, độ sâu `5000`), cây sống lưng, cây đầy đủ, ngẫu nhiên; chuỗi tăng dần `0, 1, 2, …` (đáp án `n − 1`), giá trị `0` và `10^5` xen kẽ.

In [24]:
t0 = time.perf_counter()
n = 5000
rng = random.Random(102603)
inc = TreeNode(0)
cur = inc
for i in range(1, n):
    cur.right = TreeNode(i)
    cur = cur.right
alt = TreeNode(0)
cur = alt
for i in range(1, n):
    cur.left = TreeNode(10 ** 5 if i % 2 else 0)
    cur = cur.left
for name, tr, want in [("chuỗi tăng 0..4999", inc, n - 1), ("chuỗi 0/10^5 xen kẽ", alt, 10 ** 5)]:
    res = {k: IMPLS[k](tr) for k in fast}
    assert len(set(res.values())) == 1 and res["Solution"] == want, (name, res)
    print(f"  {name:22s}: độ sâu {depth(tr)}, đáp án {res['Solution']}")
for shape in ("chain", "comb", "complete", "random"):
    tree = random_tree(rng, n, shape, 10 ** 5)
    res = {k: IMPLS[k](tree) for k in fast}
    assert len(set(res.values())) == 1, shape
    print(f"  {shape:9s}: độ sâu {depth(tree):5d}, đáp án {res['Solution']}")
print("n = 5000: mọi cài đặt (kể cả đệ quy với giới hạn tăng) khớp nhau;", round(time.perf_counter() - t0, 2), "s")

  chuỗi tăng 0..4999    : độ sâu 5000, đáp án 4999
  chuỗi 0/10^5 xen kẽ   : độ sâu 5000, đáp án 100000
  chain    : độ sâu  5000, đáp án 99938
  comb     : độ sâu  2501, đáp án 99963
  complete : độ sâu    13, đáp án 99298
  random   : độ sâu    29, đáp án 99386
n = 5000: mọi cài đặt (kể cả đệ quy với giới hạn tăng) khớp nhau; 0.15 s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chỉ so cha–con

In [25]:
t = build_tree(L1)
print("bug:", bug_parent_child_only(t), "| đúng:", Solution().maxAncestorDiff(t))
assert bug_parent_child_only(t) == 5 and Solution().maxAncestorDiff(t) == 7

bug: 5 | đúng: 7


**Vì sao sai:** tổ tiên gồm **mọi** nút trên đường lên gốc, không chỉ cha. `|8 − 1| = 7` là cặp ông–cháu.

### Lỗi 2: chỉ so với gốc

In [26]:
t = build_tree([2, 5, 0, None, None, 1])
print("bug:", bug_root_only(t), "| đúng:", Solution().maxAncestorDiff(t))
bad = next(l for l in probe if bug_root_only(build_tree(l)) != diff_brute(build_tree(l)))
print("phản ví dụ:", bad, "-> bug", bug_root_only(build_tree(bad)), "đúng", diff_brute(build_tree(bad)))
assert bug_root_only(build_tree(bad)) != diff_brute(build_tree(bad))

bug: 3 | đúng: 3
phản ví dụ: [1, None, 2, None, 0, 3] -> bug 2 đúng 3


**Vì sao sai:** cặp tốt nhất có thể không chứa gốc (tổ tiên ở giữa).

### Lỗi 3: `max − min` toàn cây

In [27]:
t = build_tree([5, 0, 10])
print("bug:", bug_global_range(t), "| đúng:", Solution().maxAncestorDiff(t))
assert bug_global_range(t) == 10 and Solution().maxAncestorDiff(t) == 5

bug: 10 | đúng: 5


**Vì sao sai:** `0` và `10` là hai nút **anh em**, không phải tổ tiên–hậu duệ.

### Lỗi 4: chỉ theo dõi `max` (hoặc chỉ `min`) của đường đi

In [28]:
t = build_tree([5, 0, 10])
print("bug (chỉ max):", bug_only_max(t), "| bug (chỉ min):", bug_include_self_in_range(t), "| đúng:", Solution().maxAncestorDiff(t))
bad = next(l for l in probe if bug_only_max(build_tree(l)) != diff_brute(build_tree(l)))
bad2 = next(l for l in probe if bug_include_self_in_range(build_tree(l)) != diff_brute(build_tree(l)))
print("phản ví dụ (max):", bad, "| (min):", bad2)

bug (chỉ max): 5 | bug (chỉ min): 5 | đúng: 5
phản ví dụ (max): [1, None, 2, None, 0, 3] | (min): [8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13]


**Vì sao sai:** nút có thể cách xa **cả** tổ tiên nhỏ nhất **hoặc** lớn nhất; phải kiểm cả hai.

### Lỗi 5: dùng chung `min/max` giữa các nhánh anh em

In [29]:
bad = next(l for l in probe if bug_shared_state(build_tree(l)) != diff_brute(build_tree(l)))
print("phản ví dụ:", bad, "| bug:", bug_shared_state(build_tree(bad)), "| đúng:", diff_brute(build_tree(bad)))
assert bug_shared_state(build_tree(bad)) != diff_brute(build_tree(bad))

phản ví dụ: [8, 3, 10, 1, 6, None, 14, None, None, 4, 7, 13] | bug: 13 | đúng: 7


**Vì sao sai:** `min/max` phải là của **đường đi** gốc → nút hiện tại; nếu không khôi phục khi quay lui, giá trị từ nhánh anh em rò sang nhánh khác. **Đúng:** truyền `(mn, mx)` theo từng nút (stack/tham số).

### Lỗi 6 (bonus): `max − min` của cây con

In [30]:
t = build_tree([5, 0, 10])
print("bug:", bug_subtree_range(t), "| đúng:", Solution().maxAncestorDiff(t))
assert bug_subtree_range(t) == 10

bug: 10 | đúng: 5


**Vì sao sai:** `mx − mn` trong một cây con có thể đến từ hai nút ở **hai nhánh khác nhau**. Chỉ được so **tổ tiên với từng hậu duệ**.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "tối ưu một hàm giữa nút và các tổ tiên (hoặc hậu duệ)" → mang tóm tắt của đường đi (ở đây `min`, `max`) xuống, hoặc trả tóm tắt của cây con lên.

**Mẹo:**

1. Hàm `|a − b|` lồi: chỉ cần `min` và `max`.
2. DFS lặp để tránh `RecursionError` với cây sâu `5000`.
3. Mỗi nút mang `(mn, mx)` **của riêng đường đi của nó** (không dùng biến toàn cục không khôi phục).
4. So con với `(mn, mx)` của **tổ tiên của nó** (đường đi tới cha).
5. Oracle: bản đồ cha (mọi tổ tiên) và đệ quy bộ lồng nhau.
6. Kiểm tra tính bất biến: mirror, dịch, co giãn.

**Họ bài cùng pattern:** Binary Tree Maximum Path Sum (124), Count Good Nodes in Binary Tree (1448), Path Sum II (113), Smallest String Starting From Leaf (988), Longest Univalue Path (687), Maximum Average Subtree (1120).

## 15. Biến thể

1. **Trả về cặp** (giá trị tổ tiên, hậu duệ) đạt đáp án.
2. **Hiệu có dấu:** `max (a.val − b.val)` với `a` tổ tiên của `b` (tổ tiên lớn hơn).
3. **Hiệu nhỏ nhất** giữa tổ tiên và hậu duệ (khác nhau) — cần thêm cấu trúc (xem kiểm bằng oracle).

Cài đặt và đối chiếu:

In [31]:
def max_signed_diff(root):
    """Biến thể 2: max(a.val - b.val), a là tổ tiên của b (a > b): theo dõi max của đường đi."""
    best = float("-inf")
    stack = [(root, root.val)]
    while stack:
        node, mx = stack.pop()
        for c in (node.left, node.right):
            if c:
                best = max(best, mx - c.val)
                stack.append((c, max(mx, c.val)))
    return best


def brute_signed(root):
    parent, order = parent_map(root)
    best = float("-inf")
    for x in order:
        y = parent[id(x)]
        while y is not None:
            best = max(best, y.val - x.val)
            y = parent[id(y)]
    return best


def min_abs_diff(root):
    """Biến thể 3: min |a - b| (tổ tiên-hậu duệ) bằng duyệt mọi tổ tiên (O(n·h))."""
    parent, order = parent_map(root)
    best = float("inf")
    for x in order:
        y = parent[id(x)]
        while y is not None:
            best = min(best, abs(x.val - y.val))
            y = parent[id(y)]
    return best


def best_pair_fast(root):
    best, pair = -1, None
    stack = [(root, root.val, root.val)]
    while stack:
        node, mn, mx = stack.pop()
        for c in (node.left, node.right):
            if c:
                for a in (mn, mx):
                    if abs(c.val - a) > best:
                        best, pair = abs(c.val - a), (a, c.val)
                stack.append((c, min(mn, c.val), max(mx, c.val)))
    return best, pair


for n in range(2, 5):
    for t in all_trees(n, (0, 3, 10)):
        root = from_tuple(t)
        assert max_signed_diff(root) == brute_signed(root)                                   # (2)
        b, pair = best_pair_fast(root)
        assert b == oracle_tuple(t) and abs(pair[0] - pair[1]) == b                          # (1)
rng = random.Random(15)
for _ in range(200):
    tr = random_tree(rng, rng.randint(2, 30), rng.choice(["random", "chain"]), 50)
    assert max_signed_diff(tr) == brute_signed(tr)
    assert min_abs_diff(tr) <= diff_brute(tr)
print("cặp đạt đáp án, hiệu có dấu: khớp oracle; hiệu nhỏ nhất ≤ hiệu lớn nhất")

cặp đạt đáp án, hiệu có dấu: khớp oracle; hiệu nhỏ nhất ≤ hiệu lớn nhất


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mọi tổ tiên | bản đồ cha | `O(n·h)` | `O(n)` | oracle |
| 2. DFS lặp `(min, max)` | mang đường đi xuống | `O(n)` | `O(n)` | **đáp án chính** |
| 3. Đệ quy dưới lên | `(min, max)` cây con | `O(n)` | `O(h)` | gọn |
| 4. BFS | trạng thái theo tầng | `O(n)` | `O(w)` | không stack sâu |
| 5. Hậu thứ tự lặp | tổ hợp từ con | `O(n)` | `O(n)` | không đệ quy |

Mọi cách khớp hai oracle trên `11 493` cây nhỏ (nhãn `{0,3,10}`) và khớp nhau tới `n = 5000`.

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [32]:
print(Solution().maxAncestorDiff(build_tree(L1)), Solution().maxAncestorDiff(build_tree(L2)), Solution().maxAncestorDiff(build_tree([5, 5])),
      Solution().maxAncestorDiff(build_tree([0, 100000])), Solution().maxAncestorDiff(build_tree([5, 0, 10])))

7 3 0 100000 5


### Bài 2 (tính tay)

`root = [2, 5, 0, null, null, 1]` (gốc `2`; con trái `5` là lá; con phải `0` có con trái `1`). (a) Liệt kê mọi cặp (tổ tiên, hậu duệ) và hiệu. (b) Đáp án? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện DFS lặp mang `(min, max)`:

In [33]:
def dfs_todo(root):
    best = 0
    stack = [(root, root.val, root.val)]
    while stack:
        node, mn, mx = stack.pop()
        for child in (node.left, node.right):
            if child:
                pass        # TODO 1: best = max(best, abs(child.val - mn), abs(child.val - mx))
                pass        # TODO 2: stack.append((child, min(mn, child.val), max(mx, child.val)))
    return best


# Khi làm xong, bỏ comment để kiểm tra:
# assert dfs_todo(build_tree(L1)) == 7 and dfs_todo(build_tree(L2)) == 3

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **BFS** (hàng đợi `(nút, mn, mx)`):

In [34]:
class MySolution:
    def maxAncestorDiff(self, root):
        pass            # TODO


# Khi làm xong:
# assert MySolution().maxAncestorDiff(build_tree(L1)) == 7 and MySolution().maxAncestorDiff(build_tree(L2)) == 3

### Bài 5 (mini challenge): trả về cặp

Viết `best_pair_values(root)` trả về `(a, b)`: giá trị của một cặp (tổ tiên `a`, hậu duệ `b`) đạt `|a − b|` lớn nhất. Kiểm bằng cách xác minh `a` là tổ tiên của `b` (dùng bản đồ cha) và `|a − b|` bằng đáp án.

In [35]:
def best_pair_values(root):
    pass            # TODO


# Khi làm xong:
# a, b = best_pair_values(build_tree(L1)); assert abs(a - b) == 7

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: ví dụ 1, 2 cho `7`, `3`.
- Mức 2: hai nút bằng nhau cho `0`; `[0, 100000]` cho `100000`; `[5,0,10]` chỉ so gốc với con ⇒ `5`.
- Mức 3: `7 3 0 100000 5`.

**Bài 2**

- Mức 1: các cặp: `(2,5)`, `(2,0)`, `(2,1)`, `(0,1)`.
- Mức 2: hiệu `3`, `2`, `1`, `1`.
- Mức 3: đáp án `3`.

**Bài 3**

- Mức 1: so con với `mn`, `mx` của tổ tiên.
- Mức 2: `best = max(best, abs(child.val - mn), abs(child.val - mx))`.
- Mức 3: đẩy `(child, min(mn, child.val), max(mx, child.val))`.

**Bài 4**

- Mức 1: `queue = deque([(root, root.val, root.val)])`.
- Mức 2: `popleft`, duyệt con, cập nhật `best`.
- Mức 3: đẩy trạng thái mới; trả `best`.

**Bài 5**

- Mức 1: khi cập nhật `best`, ghi lại cặp `(mn hoặc mx, child.val)` đạt hiệu lớn nhất.
- Mức 2: xét cả hai ứng viên `mn` và `mx`.
- Mức 3: trả cặp tương ứng khi kết thúc duyệt.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [36]:
vals = (Solution().maxAncestorDiff(build_tree(L1)), Solution().maxAncestorDiff(build_tree(L2)), Solution().maxAncestorDiff(build_tree([5, 5])),
        Solution().maxAncestorDiff(build_tree([0, 100000])), Solution().maxAncestorDiff(build_tree([5, 0, 10])))
print(vals)
assert vals == (7, 3, 0, 100000, 5)
print("Bài 1 OK")

(7, 3, 0, 100000, 5)
Bài 1 OK


### Bài 2

In [37]:
t = build_tree([2, 5, 0, None, None, 1])
parent, order = parent_map(t)
pairs = []
for x in order:
    y = parent[id(x)]
    while y is not None:
        pairs.append((y.val, x.val, abs(y.val - x.val)))
        y = parent[id(y)]
print(sorted(pairs))
assert sorted(pairs) == [(0, 1, 1), (2, 0, 2), (2, 1, 1), (2, 5, 3)]
assert Solution().maxAncestorDiff(t) == 3
print("Bài 2 OK: đáp án 3")

[(0, 1, 1), (2, 0, 2), (2, 1, 1), (2, 5, 3)]
Bài 2 OK: đáp án 3


### Bài 3

In [38]:
def dfs_answer(root):
    best = 0
    stack = [(root, root.val, root.val)]
    while stack:
        node, mn, mx = stack.pop()
        for child in (node.left, node.right):
            if child:
                best = max(best, abs(child.val - mn), abs(child.val - mx))
                stack.append((child, min(mn, child.val), max(mx, child.val)))
    return best


assert dfs_answer(build_tree(L1)) == 7 and dfs_answer(build_tree(L2)) == 3
for n in range(2, 5):
    for t in all_trees(n, (0, 3, 10)):
        assert dfs_answer(from_tuple(t)) == oracle_tuple(t)
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [39]:
class MySolutionAnswer:
    def maxAncestorDiff(self, root):
        best = 0
        queue = deque([(root, root.val, root.val)])
        while queue:
            node, mn, mx = queue.popleft()
            for c in (node.left, node.right):
                if c:
                    best = max(best, abs(c.val - mn), abs(c.val - mx))
                    queue.append((c, min(mn, c.val), max(mx, c.val)))
        return best


assert MySolutionAnswer().maxAncestorDiff(build_tree(L1)) == 7 and MySolutionAnswer().maxAncestorDiff(build_tree(L2)) == 3
for n in range(2, 5):
    for t in all_trees(n, (0, 3, 10)):
        assert MySolutionAnswer().maxAncestorDiff(from_tuple(t)) == oracle_tuple(t)
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [40]:
def best_pair_values(root):
    best, pair = -1, None
    stack = [(root, root.val, root.val)]
    while stack:
        node, mn, mx = stack.pop()
        for c in (node.left, node.right):
            if c:
                for a in (mn, mx):
                    if abs(c.val - a) > best:
                        best, pair = abs(c.val - a), (a, c.val)
                stack.append((c, min(mn, c.val), max(mx, c.val)))
    return pair


a, b = best_pair_values(build_tree(L1))
assert abs(a - b) == 7
rng = random.Random(5)
for _ in range(300):
    tr = random_tree(rng, rng.randint(2, 40), rng.choice(["random", "chain", "comb"]), 100)
    a, b = best_pair_values(tr)
    parent, order = parent_map(tr)
    ok = False
    for x in order:
        if x.val == b:
            y = parent[id(x)]
            while y is not None:
                if y.val == a:
                    ok = True
                y = parent[id(y)]
    assert ok and abs(a - b) == diff_brute(tr)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (tối ưu giữa nút và tổ tiên → mang `(min, max)` của đường đi xuống):**

```python
class Solution:
    def maxAncestorDiff(self, root: Optional[TreeNode]) -> int:
        best = 0
        stack = [(root, root.val, root.val)]
        while stack:
            node, mn, mx = stack.pop()
            for child in (node.left, node.right):
                if child:
                    best = max(best, abs(child.val - mn), abs(child.val - mx))
                    stack.append((child, min(mn, child.val), max(mx, child.val)))
        return best
```

**Checklist trước khi submit:**

- [ ] So con với `mn`, `mx` của **mọi tổ tiên** (đường đi tới cha), không chỉ cha hay gốc?
- [ ] Mỗi nút mang `(mn, mx)` riêng của đường đi của nó (không biến toàn cục không khôi phục)?
- [ ] Không dùng `max − min` toàn cây hay của cây con (có thể là hai nút anh em)?
- [ ] Cây có thể sâu `5000`: dùng DFS lặp hoặc tăng giới hạn đệ quy?
- [ ] Đã thử `[8,3,10,1,6,null,14,null,null,4,7,13] → 7`, `[1,null,2,null,0,3] → 3`, `[5,0,10] → 5`?
- [ ] Hai nút (gốc + một con) cho `|gốc − con|`?
- [ ] Đúng tên `class Solution`, `maxAncestorDiff(self, root)`, không in debug.